# Frequent pattern mining

_Part of the **Decay Before Archival** project — see `README.md` for full project context. Reads the merged tables written by `integrate_datasets.ipynb`; no BigQuery queries._

**Purpose:** the second half of the progress report — *frequent pattern mining*. Every variable is turned into binary items (one-hot for categoricals, an inequality for continuous variables), then `apriori`, `fpgrowth` and `association_rules` from `mlxtend.frequent_patterns` are run at several `min_support` values.

Two tables, mined separately (Option 3 in `integrate_datasets.ipynb`):

| Table | File | Rows | Outcome item |
|---|---|---|---|
| A: GH Archive + deps.dev + Scorecard | `data/merged_dataset.csv` | 62,627 | `inactive` / `active` (from `inactive_next_365d`) |
| B: GitHub API repos | `data/github_api_table.csv` | 10,330 | `archived` |

**Reading the rules:** the outcome is very unbalanced (84.9% of Table A repos are inactive, 3.1% of Table B repos are archived), so *confidence* alone is misleading — any rule ending in `inactive` has ~85% confidence by default. Rules are ranked by **lift** (confidence ÷ base rate of the consequent).

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

pd.set_option("display.max_colwidth", 120)
# mlxtend computes a "certainty" metric that divides by zero for rules with confidence 1; we don't use it.
warnings.filterwarnings("ignore", message="invalid value encountered in divide", category=RuntimeWarning)
DATA = Path("../data")
FIG_DIR = Path("../figures")

def save_svg(fig, name):
    """Write a figure to figures/ as SVG (no timestamp, so re-runs give clean diffs)."""
    fig.savefig(FIG_DIR / name, bbox_inches="tight", metadata={"Date": None})

A = pd.read_csv(DATA / "merged_dataset.csv")
B = pd.read_csv(DATA / "github_api_table.csv")
print(f"Table A: {len(A):,} rows | Table B: {len(B):,} rows")

## 1. Binarize Table A

Thresholds come from the distributions in the source notebooks: the median repo has 1 actor, 1 active month and 3 pushes, so most cutoffs are "more than the typical tiny repo" (`> 0`, `> 1`, `>= 3`, `>= 10`). Missing values become `False` (e.g. no Scorecard score → `scorecard>=4` is False); the `in_*` items keep "not in this source" visible as its own item.

| Item | Definition | Source |
|---|---|---|
| `no_push_2024` | no push events in 2024 | GH Archive |
| `no_push_h2` | pushed in H1 but not in H2 2024 (stopped mid-year) | GH Archive |
| `pushed_last_30d` | last push ≤ 30 days before the cutoff | GH Archive |
| `new_in_h2` | no qualifying events in H1 2024 (repo started in H2) | GH Archive |
| `declining_h2_vs_h1` | fewer qualifying events in H2 than H1 | GH Archive |
| `pushes>=10` | at least 10 push events in 2024 | GH Archive |
| `active_months>=3` | events in at least 3 months of 2024 | GH Archive |
| `multi_actor` | more than one distinct actor in 2024 | GH Archive |
| `has_pr_2024` / `has_issue_2024` / `has_release_2024` | at least one such event | GH Archive |
| `starred_2024` / `forked_2024` | received a star / fork in 2024 | GH Archive |
| `in_depsdev` / `has_package` | known to deps.dev / publishes a package | deps.dev |
| `released_pkg_2024` | published a package version in 2024 | deps.dev |
| `in_scorecard` / `scorecard>=4` | has a Scorecard score / score at least 4 | Scorecard |
| `inactive` / `active` | `inactive_next_365d` = 1 / 0 | label |

In [ ]:
items_a = pd.DataFrame({
    "no_push_2024": A["push_events_2024"].eq(0),
    "no_push_h2": A["push_events_h1_2024"].gt(0) & A["push_events_h2_2024"].eq(0),
    "pushed_last_30d": A["days_since_last_push_at_cutoff"].le(30),
    "new_in_h2": A["qualifying_events_h1_2024"].eq(0),
    "declining_h2_vs_h1": A["qualifying_events_h2_2024"].lt(A["qualifying_events_h1_2024"]),
    "pushes>=10": A["push_events_2024"].ge(10),
    "active_months>=3": A["active_months_2024"].ge(3),
    "multi_actor": A["distinct_actors_2024"].gt(1),
    "has_pr_2024": A["pull_request_events_2024"].gt(0),
    "has_issue_2024": A["issue_events_2024"].gt(0),
    "has_release_2024": A["release_events_2024"].gt(0),
    "starred_2024": A["stars_received_2024"].gt(0),
    "forked_2024": A["fork_events_2024"].gt(0),
    "in_depsdev": A["in_depsdev"].eq(1),
    "has_package": A["has_package"].eq(1),
    "released_pkg_2024": A["versions_2024"].gt(0),
    "in_scorecard": A["in_scorecard"].eq(1),
    "scorecard>=4": A["scorecard_score"].ge(4),  # NaN >= 4 is False
    "inactive": A["inactive_next_365d"].eq(1),
    "active": A["inactive_next_365d"].eq(0),
})
assert items_a.dtypes.eq(bool).all() and len(items_a) == len(A)

support_a = items_a.mean().sort_values(ascending=False).to_frame("support")
support_a["repos"] = items_a.sum()
support_a.round(4)

**Notes:**

- 20 items. Supports range from 84.9% (`inactive`) down to 0.22% (`scorecard>=4`, 137 repos).
- Two items dominate: `inactive` (84.9%) and `new_in_h2` (82.7%). At high `min_support` almost every frequent itemset is built from them.
- The deps.dev and Scorecard items are all below 2.3%, so they can only appear at `min_support` ≤ 0.02 (and in combinations only at ≤ 0.005). Section 4 mines them on their own.
- `no_push_h2` is defined from the H1/H2 push counts rather than "last push > 180 days ago". A day-count cutoff also catches repos whose last push was on July 1–4, which is already H2 (898 of them have no H1 activity at all), and gives contradictory rules like "no H1 events and no push for 180 days".

## 2. `apriori` vs `fpgrowth` while varying `min_support`

Both algorithms must find exactly the same frequent itemsets; they differ only in speed. Rules are counted at a 0.6 confidence floor.

In [ ]:
def as_set(freq):
    """Itemsets with rounded supports, for comparing the two algorithms."""
    return {(frozenset(s), round(v, 10)) for s, v in zip(freq["itemsets"], freq["support"])}

def sweep(items, supports, max_len=None):
    rows = []
    for s in supports:
        t = time.perf_counter(); ap = apriori(items, min_support=s, use_colnames=True, max_len=max_len, low_memory=True); t_ap = time.perf_counter() - t
        t = time.perf_counter(); fp = fpgrowth(items, min_support=s, use_colnames=True, max_len=max_len); t_fp = time.perf_counter() - t
        rules = association_rules(fp, metric="confidence", min_threshold=0.6) if len(fp) > 1 else []
        rows.append({"min_support": s, "itemsets": len(fp), "same_itemsets": as_set(ap) == as_set(fp),
                     "rules_conf>=0.6": len(rules), "max_itemset_size": int(fp["itemsets"].map(len).max()),
                     "apriori_s": round(t_ap, 3), "fpgrowth_s": round(t_fp, 3)})
    return pd.DataFrame(rows).set_index("min_support")

SUPPORTS = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]
sweep_a = sweep(items_a, SUPPORTS)
assert sweep_a["same_itemsets"].all()
sweep_a

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = sweep_a.index
axes[0].plot(x, sweep_a["itemsets"], "o-", color="#4C78A8", label="frequent itemsets")
axes[0].plot(x, sweep_a["rules_conf>=0.6"], "s-", color="#F58518", label="rules (confidence ≥ 0.6)")
axes[0].set(xscale="log", yscale="log", xlabel="min_support (log)", ylabel="count (log)", title="Table A: output size vs min_support")
axes[0].invert_xaxis(); axes[0].legend()
axes[1].plot(x, sweep_a["apriori_s"], "o-", color="#E45756", label="apriori")
axes[1].plot(x, sweep_a["fpgrowth_s"], "s-", color="#54A24B", label="fpgrowth")
axes[1].set(xscale="log", xlabel="min_support (log)", ylabel="seconds", title="Table A: runtime vs min_support")
axes[1].invert_xaxis(); axes[1].legend()
for ax in axes: ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save_svg(fig, "mining_support_sweep.svg"); plt.show()

**Assignment item:** Frequent Pattern Mining → *what happens when you vary `min_support`?*

**What it shows:**

- **Output size grows very fast as `min_support` drops.** Itemsets go from 3 at 0.5 to 1,609 at 0.005, and rules with confidence ≥ 0.6 go from 2 to 9,967. The largest itemset grows from 2 items to 8.
- **At 0.5, the only rule is the trivial `new_in_h2 → inactive`:** most repos are new, and most repos go inactive. Interesting combinations only appear below 0.05.
- **Below 0.02, rules outgrow itemsets.** Every larger itemset produces many rules, mostly re-combinations of the same correlated activity flags. That is why the rules below are filtered by lift and pruned for redundancy.
- **`apriori` and `fpgrowth` return identical itemsets at every level** (checked by the `assert`).
- **Runtime:** with only 20 items both finish well under a second. FP-Growth has a fixed setup cost, so `apriori` (with `low_memory=True`) is as fast or faster here. The two only meet around 0.005. FP-Growth's advantage would show with many more items or lower support. Exact timings vary from run to run.

## 3. Rules that predict the label (Table A)

`min_support = 0.005` (≈ 313 repos) is the lowest level that still runs in seconds and keeps every source's items in play. Only rules whose consequent is exactly the outcome item are kept, antecedents are capped at 3 items to stay readable, and redundant rules are dropped when a shorter antecedent already has the same or higher lift.

In [ ]:
def outcome_rules(freq, outcome, max_ante=3):
    rules = association_rules(freq, metric="lift", min_threshold=1.0)
    r = rules.loc[rules["consequents"].eq(frozenset([outcome])) & rules["antecedents"].map(len).le(max_ante)].copy()
    r = r.sort_values(["lift", "support"], ascending=False)
    # Drop a rule if a strict subset of its antecedent already reaches at least the same lift.
    kept = []
    for ante, lift in zip(r["antecedents"], r["lift"]):
        kept.append(not any(k < ante and l >= lift - 1e-9 for k, l in zip(r["antecedents"], r["lift"])))
    r = r.loc[kept]
    r.insert(0, "rule", r["antecedents"].map(lambda s: " & ".join(sorted(s))) + f"  →  {outcome}")
    return r[["rule", "support", "confidence", "lift", "leverage"]].reset_index(drop=True)

freq_a = fpgrowth(items_a, min_support=0.005, use_colnames=True)
print(f"{len(freq_a):,} frequent itemsets at min_support=0.005 | base rates: inactive {items_a['inactive'].mean():.3f}, active {items_a['active'].mean():.3f}")

rules_active = outcome_rules(freq_a, "active")
rules_inactive = outcome_rules(freq_a, "inactive")
print(f"rules → active: {len(rules_active)} | rules → inactive: {len(rules_inactive)}")
display(rules_active.head(12).round(3))
rules_inactive.head(12).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for r, color, label in [(rules_active, "#2E7D68", "→ active"), (rules_inactive, "#C85A54", "→ inactive")]:
    sc = ax.scatter(r["support"], r["confidence"], s=12 + 8 * r["lift"], color=color, alpha=0.6, label=f"{label} ({len(r)} rules)")
ax.axhline(items_a["inactive"].mean(), color="#C85A54", ls="--", lw=1)
ax.axhline(items_a["active"].mean(), color="#2E7D68", ls="--", lw=1)
ax.text(ax.get_xlim()[1], items_a["inactive"].mean(), " base rate inactive", va="center", fontsize=8)
ax.text(ax.get_xlim()[1], items_a["active"].mean(), " base rate active", va="center", fontsize=8)
ax.set(xscale="log", xlabel="support (log)", ylabel="confidence", title="Table A: rules with the label as consequent (marker size = lift)")
ax.legend(loc="center right"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save_svg(fig, "mining_label_rules_tableA.svg"); plt.show()

**Assignment item:** Frequent Pattern Mining → *apply `association_rules`; any interesting patterns?*

**What it shows:**

- **→ active: 181 non-redundant rules, lift up to 6.4.**
  - Confidence goes up to 96%, against a 15% base rate.
  - Every top rule combines `pushed_last_30d` with a sign of outside engagement: issues, forks, stars, several actors, a release, or 3+ active months. For example, `forked_2024 & has_issue_2024 & pushed_last_30d → active` has 96.3% confidence and lift 6.36.
  - The broadest strong rule is `active_months>=3 & pushed_last_30d & starred_2024 → active` (1.2% of repos, 90% confidence).
- **A mid-year dip is not a death signal.** `declining_h2_vs_h1` appears in active rules when combined with a recent push and stars, PRs or several actors.
- **→ inactive: only 5 rules survive, lift ≤ 1.09.**
  - With an 85% base rate there is little room above it.
  - The best is `no_push_h2 & pushes>=10 → inactive` (9.0% of repos, 92% confidence, lift 1.085): repos that were busy in H1 and then stopped pushing in H2.
  - `new_in_h2 → inactive` (88%) covers 73% of all repos: young, short-lived projects.
- **No deps.dev or Scorecard items** appear in the label rules at this support level; their support is too low in the full table.

## 4. Packaged repos only (deps.dev + Scorecard items)

Only 1.5% of Table A has a package, so deps.dev and Scorecard items never combine with much else at a support that keeps the full table readable. Mining the 951 packaged repos on their own lets those items compete, and the label is balanced here (44.9% inactive), so lift is easier to read.

In [ ]:
pkg = items_a.loc[items_a["has_package"]].drop(columns=["in_depsdev", "has_package"])
print(f"{len(pkg):,} packaged repos | inactive rate {pkg['inactive'].mean():.3f}")
display(sweep(pkg, [0.5, 0.2, 0.1, 0.05]))

freq_pkg = fpgrowth(pkg, min_support=0.05, use_colnames=True)
pkg_active = outcome_rules(freq_pkg, "active")
pkg_inactive = outcome_rules(freq_pkg, "inactive")
print(f"rules → active: {len(pkg_active)} | rules → inactive: {len(pkg_inactive)}")
display(pkg_active.head(10).round(3))
pkg_inactive.head(10).round(3)

**Notes:**

- **The output explodes faster here.** Items are much more common and strongly correlated among packaged repos, so at the same `min_support` = 0.05 there are 5,995 itemsets (vs 61 in the full table) and about 170k rules at confidence ≥ 0.6.
- **→ active:**
  - The top rule is `has_issue_2024 & has_pr_2024 & pushed_last_30d → active`: 13.1% of packaged repos (about 125), 98.4% confidence, lift 1.79.
  - Scorecard now matters: `multi_actor & pushed_last_30d & scorecard>=4 → active` has 98.3% confidence.
- **→ inactive:**
  - `in_scorecard & new_in_h2 → inactive` (68.1%, lift 1.52): established, scored repos that had no events in H1 2024 are the riskiest packaged repos.
  - `no_push_h2 & released_pkg_2024 → inactive` (64.8%, lift 1.44): publishing a package version in 2024 does not protect a repo if pushes stopped in H2.
- Lifts are lower than in the full table (max ~1.8) because the base rate is balanced (55% active). Confidence is far higher, though: up to 98% for active and 68% for inactive.

## 5. Table B: GitHub API repos (outcome `archived`)

Table B has categorical variables, so `language` and `license_spdx` are one-hot encoded (`pd.get_dummies`), keeping the 6 most common languages and 5 most common licenses and grouping the rest as `other` (missing language becomes `unknown`). Continuous metrics are cut with inequalities. `decay_stage` and `archetype` are left out: they are derived from the same commit/release columns, so they would only produce circular rules.

Caveat: `archived` and `days_since_last_commit` were both observed at collection time (2026), and an archived repo cannot receive commits, so "no commit for a year → archived" is partly mechanical. It is kept as an item but read with that in mind.

In [ ]:
top_lang = B["language"].value_counts().index[:6]
top_lic = B["license_spdx"].value_counts().index[:5]
lang = B["language"].where(B["language"].isin(top_lang), np.where(B["language"].isna(), "unknown", "other"))
lic = B["license_spdx"].where(B["license_spdx"].isin(top_lic), "other")
onehot = pd.concat([pd.get_dummies(lang, prefix="lang"), pd.get_dummies(lic, prefix="license")], axis=1)

items_b = pd.concat([onehot, pd.DataFrame({
    "stars>=2000": B["stargazers_count"].ge(2000),  # pool minimum is 470 stars, median 1,014
    "contributors>=5": B["n_contributors_top100"].ge(5),
    "bus_factor<=1": B["bus_factor"].le(1),
    "has_merged_pr": B["merged_prs"].gt(0),
    "has_opened_issue": B["opened_issues"].gt(0),
    "resolves_issues": (B["resolved_issues"] / B["opened_issues"]).ge(0.5),  # 0/0 -> NaN -> False
    "never_released": B["last_release_at"].isna(),
    "release>365d": B["days_since_last_release"].gt(365),
    "commit<=30d": B["days_since_last_commit"].le(30),
    "commit>365d": B["days_since_last_commit"].gt(365),
    "archived": B["archived"].astype(bool),
})], axis=1)
assert items_b.dtypes.eq(bool).all()
display(items_b.mean().sort_values(ascending=False).round(4).to_frame("support").T)

sweep_b = sweep(items_b, [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005], max_len=4)
assert sweep_b["same_itemsets"].all()
sweep_b

In [ ]:
freq_b = fpgrowth(items_b, min_support=0.003, use_colnames=True, max_len=4)
rules_archived = outcome_rules(freq_b, "archived")
print(f"{len(freq_b):,} itemsets at min_support=0.003 | base rate archived {items_b['archived'].mean():.3f} | rules → archived: {len(rules_archived)}")
rules_archived.head(15).round(3)

**Notes:**

- **Items:** one-hot language (top 6, plus `other` and `unknown`) and license (top 5, plus `other`), plus 11 threshold items including `archived`. Every repo has ≥ 470 stars (a search-pool filter), so popularity is cut at `stars>=2000` (roughly the top quartile).
- **Sweep:** itemsets grow from 3 at 0.5 to 2,174 at 0.005, and `apriori` and `fpgrowth` agree at every level. `max_len=4` keeps the rule count readable; without it the one-hot columns multiply the combinations.
- **→ archived: 20 non-redundant rules at `min_support` = 0.003 (≈ 31 repos).**
  - The top rule is `commit>365d & contributors>=5 & release>365d → archived`: confidence 14.9%, lift 4.85.
  - On its own, `commit>365d` has lift 2.2 and `release>365d` has lift 2.9.
  - Confidence stays low because only 3.1% of repos are archived. These rules are **warning signs, not predictions**.
- **Who gets formally archived:**
  - `contributors>=5` appears in most top rules, while `bus_factor<=1` never does. Larger, multi-contributor projects get formally archived once they stall. Solo repos are more likely to just be abandoned without the archive flag.
  - `stars>=2000` never appears, so popularity does not protect against archival.
  - TypeScript (lift 1.34) and GPL-3.0 (lift 1.21) are slightly over-represented.
- **Caveat:** the commit-age rules are partly mechanical, as explained above (archived repos cannot receive commits).

## 6. Summary and strategic insights

1. **Healthy repos are easier to recognise than doomed ones.**
   - In Table A, rules that predict *staying active* reach lift ~6.4 and 96% confidence.
   - Rules that predict *inactivity* barely beat the 85% base rate (lift ≤ 1.09).
   - So a practical health check should flag repos that are **missing** the healthy pattern (a recent push plus issues, PRs, forks or several contributors), rather than search for a specific decay pattern.
2. **The clearest early warning is a mid-year stop.** `no_push_h2` (busy in H1, no pushes in H2) is the strongest inactivity item, both in the full table (92% inactive) and among packaged repos (lift 1.44–1.50). A drop in activity *with* recent pushes and community engagement is not a warning sign.
3. **Publishing a package is not the same as being maintained.** Packaged repos that released in 2024 but stopped pushing in H2 are 65% inactive the next year, against a 45% base rate for packaged repos.
4. **Scorecard adds signal for dependency choices.** Among packaged repos, `scorecard>=4` combined with recent pushes and several contributors gives 98% confidence of staying active. For choosing dependencies, prefer packages with a recent push, more than one contributor and a Scorecard score of at least 4.
5. **Effect of `min_support`:**
   - High support (≥ 0.2) only returns trivial rules driven by the dominant items (`new_in_h2 → inactive`).
   - The source-specific items (deps.dev, Scorecard) only appear at ≤ 0.005 or after subsetting to packaged repos.
   - Below 0.02 the number of rules grows much faster than the number of itemsets, so lift ranking and redundancy pruning are needed to keep the output usable.
6. **Formal archival (Table B) comes after a year without commits or releases, mostly in multi-contributor projects.** Popularity (≥ 2,000 stars) does not prevent it. Because only 3.1% of repos are archived, these rules are warning signs (lift up to 4.9), not predictions.